In [1]:
import random
import string
import numpy as np
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)
from pymilvus.model.hybrid import BGEM3EmbeddingFunction

In [2]:
import torch
print(torch.cuda.is_available())  # This should return True if GPU is available
print(torch.cuda.device_count())  # This returns the number of available GPUs
print(torch.cuda.current_device())  # This gives the index of the current GPU
print(torch.cuda.get_device_name(0))  # This returns the name of the first GPU


True
1
0
NVIDIA GeForce GTX 1080


In [3]:
from FlagEmbedding import BGEM3FlagModel


In [4]:
import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Add the EDP directory to the Python path
# sys.path.append(os.path.abspath(os.path.join('.', 'NIR')))

In [5]:
# from parser_2 import Parser2
from parser_2_v2 import Parser2

In [6]:
import json
import gzip
def load_compressed_json(file_path):
    """Function to load data from a compressed JSON file."""
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        return json.load(f)

def load_all_data_into_list(parsed_directory):
    """Load data from all compressed JSON files into one list."""
    all_data = []
    # Walk through the parsed directory to find compressed JSON files
    for root, dirs, files in os.walk(parsed_directory):
        for file in files:
            if file.endswith('.json.gz'):
                file_path = os.path.join(root, file)
                print(f"Loading data from {file_path}...")
                data = load_compressed_json(file_path)
                all_data.extend(data)

    return all_data

# Usage example
parsed_directory_path = '/home/sbasir/Thesis/Thesis/cp'
all_data = load_all_data_into_list(parsed_directory_path)

def merge_text_fields(data):
    for item in data:
        # Merge the fields into 'text', separating them by " | "
        merged_text = " | ".join(filter(None, [item.get('text', ''), 
                                               item.get('provided_data', ''), 
                                               item.get('enriched_data', ''), 
                                               item.get('translated_data', '')]))
        
        # Assign the merged text back to the 'text' field
        item['text'] = merged_text
        
        # Remove the individual fields as they're now part of 'text'
        item.pop('provided_data', None)
        item.pop('enriched_data', None)
        item.pop('translated_data', None)
    
    return data

data = merge_text_fields(all_data)

Loading data from /home/sbasir/Thesis/Thesis/cp/5/5.json.gz...
Loading data from /home/sbasir/Thesis/Thesis/cp/1/1.json.gz...
Loading data from /home/sbasir/Thesis/Thesis/cp/10/10.json.gz...


In [7]:
only_text = [x['text'] for x in data]

In [8]:
# Generate embeddings using BGEM3 model
ef = BGEM3EmbeddingFunction(use_fp16=False, device="cuda")
embeddings = ef(only_text)
# Debug prints to verify the dimensions
print(f"Number of texts: {len(only_text)}")
print(f"Dense embeddings shape: {len(embeddings['dense'])}")
print(f"Sparse embeddings shape: {(embeddings['sparse']).shape}")

# Prepare data for insertion
# entities = [
#     [item['id'] for item in data],  # IDs
#     only_text,  # Texts
#     embeddings["dense"],  # Dense vectors
#     embeddings["sparse"]  # Sparse vectors
# ]
# entities = [
#     {
#         "id": item['id'],
#         "text": text,
#         "dense_vector": dense,
#         "sparse_vector": sparse
#     }
#     for item, text, dense, sparse in zip(data, only_text, embeddings["dense"], embeddings["sparse"])
# ]

# # Verify the lengths of each component to ensure they match
# print(f"Length of IDs: {len(entities[0])}")
# print(f"Length of texts: {len(entities[1])}")
# print(f"Shape of dense vectors: {len(entities[2])}")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

/home/sbasir/Thesis/myenv3.10/lib/python3.10/site-packages/FlagEmbedding/BGE_M3/modeling.py:335: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  colbert_state_dict = torch.loa

Number of texts: 835
Dense embeddings shape: 835
Sparse embeddings shape: (835, 250002)


NameError: name 'entities' is not defined

In [20]:
entities = [
    {
        "id": item['id'],
        "text": text,
        "dense_vector": dense,
        "sparse_vector": sparse
    }
    for item, text, dense, sparse in zip(data, only_text, embeddings["dense"], embeddings["sparse"])
]

entities2 = [
    [item['id'] for item in data],  # IDs
    only_text,  # Texts
    embeddings["dense"],  # Dense vectors
    embeddings["sparse"]  # Sparse vectors
]

In [16]:
from pymilvus import connections

connections.connect("default", host="localhost", port="19530")

if connections.has_connection("default"):
    print("Successfully connected to Milvus")
else:
    print("Failed to connect to Milvus")


Successfully connected to Milvus


In [17]:
# Define the data schema for the new Collection
fields = [
    # Use provided id as primary key
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, max_length=100),
    # Store the original text
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
    # Store dense vectors
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=1024),  # Ensure the dimension matches your embeddings
    # Store sparse vectors
    FieldSchema(name="sparse_vector", dtype=DataType.SPARSE_FLOAT_VECTOR),  
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'hybrid_dem3'
col = Collection(col_name, schema, consistency_level="Strong")

In [18]:
sparse_index = {"index_type": "SPARSE_INVERTED_INDEX", "metric_type": "IP"}
col.create_index("sparse_vector", sparse_index)
dense_index = {"index_type": "FLAT", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
# col.load()

Status(code=0, message=)

In [ ]:
# if col.has_collection(col_name):
#     print("Collection exists.")
    
# # # Create an index for the dense vector field
# # dense_index = client.prepare_index_params(index_type="IVF_FLAT", metric_type="L2", params={"nlist": 1024})

# # client.create_index(col_name, dense_index)

In [ ]:
# sparse_index = client.prepare_index_params(index_type="FLAT", metric_type="JACCARD", params={"nlist": 1024})
# client.create_index("sparse_vector", sparse_index)

In [21]:
col.insert(entities2)

(insert count: 835, delete count: 0, upsert count: 0, timestamp: 452771150196637700, success count: 835, err count: 0

In [16]:
col.flush()

In [17]:
query = "1867-08-30 - Seren Cymru"
query_embeddings = ef([query])
k=10
# Prepare the search requests for both vector fields
sparse_search_params = {"metric_type": "IP"}
sparse_req = AnnSearchRequest(query_embeddings["sparse"],
                              "sparse_vector", sparse_search_params, limit=k)
dense_search_params = {"metric_type": "IP"}
dense_req = AnnSearchRequest(query_embeddings["dense"],
                             "dense_vector", dense_search_params, limit=k)

# Search topK docs based on dense and sparse vectors and rerank with RRF.
res = col.hybrid_search([sparse_req, dense_req], rerank=RRFRanker(),
                        limit=k, output_fields=['text'])

for result in res[0]:
    print(result)

KeyboardInterrupt: 

In [30]:
# Assuming `col` is your Milvus collection object
num_vectors = col.num_entities

print(f"The collection contains {num_vectors} vectors.")

The collection contains 835 vectors.
